In [ ]:
    ############    #############   Concurrency vs Parallelism   #############   ##############   

 =>  Concurrency: structuring a program so multiple tasks can make progress by interleaving
       (asyncio, one thread, cooperative switching on I/O waits).

 =>  Parallelism: tasks literally executing at the same instant on multiple CPU cores
       (multiprocessing, or native threads for non-Python/C-extension work).

 =>  Python's GIL (Global Interpreter Lock) means only one thread runs Python bytecode at a
       time, so threading gives concurrency for I/O-bound work but not parallelism for
       CPU-bound work.

 =>  Decision rule:
        * I/O-bound (network, disk, waiting on external services)  -> asyncio or threads
        * CPU-bound (parsing, hashing, embedding math, image work)  -> multiprocessing


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def cpu_bound_work(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(cpu_bound_work, [N] * 4))
print(f"threads (CPU-bound): {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as pool:
    list(pool.map(cpu_bound_work, [N] * 4))
print(f"processes (CPU-bound): {time.perf_counter() - start:.2f}s")


In [ ]:
 =>  Expect the ThreadPoolExecutor run to be about as slow as running everything on one
       thread (the GIL serializes the Python bytecode), while ProcessPoolExecutor actually
       uses multiple cores and finishes faster for this CPU-bound loop -- when it runs at all
       (see the note below).

 =>  Flip the workload to something I/O-bound (e.g. requests with network calls or
       asyncio.sleep) and threads/async win instead, because the GIL is released while
       waiting on I/O.

 =>  IMPORTANT -- if the ProcessPoolExecutor cell raised a RuntimeError about
       'bootstrapping phase' / BrokenProcessPool: on macOS and Windows, multiprocessing
       defaults to the 'spawn' start method, which re-imports your main module in each
       child process. A worker function defined directly in a notebook cell (or an
       unguarded script) can't be re-imported that way, and process creation fails.
       Fix: move cpu_bound_work into its own real .py file and import it, or run the
       ProcessPoolExecutor code from a script guarded with:
           if __name__ == '__main__':
               ...
       ThreadPoolExecutor doesn't have this problem -- threads share the same process, no
       re-import needed, which is one more reason threads are simpler to reach for even
       though they don't give you parallelism for CPU-bound work.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Rerun the demo with a genuinely I/O-bound function (e.g. time.sleep or a real
           network call) instead of cpu_bound_work, and confirm threads now win.

 =>  [ ] Profile one real function in your own codebase -- is it actually CPU-bound or
           I/O-bound? Pick the matching concurrency tool.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Reaching for multiprocessing by default 'for speed' on I/O-bound work -- processes have
       real overhead (separate memory, serialization to pass data) that isn't worth paying
       when the bottleneck is waiting on a network call.

 =>  Assuming more threads always helps -- past a point, GIL contention and context-switch
       overhead make CPU-bound threaded code slower, not faster.

 =>  Defining a ProcessPoolExecutor's worker function inside a notebook/script's main scope
       on macOS/Windows -- the 'spawn' start method needs to re-import it in each child
       process, which fails for anything not defined in a real, importable module.
